# 07e — Semantic Alignment + TPI

## Objective

This notebook integrates the semantic alignment developed in **07d** directly with the
**Text Preserved Information (TPI)** measure from **07a/07b**.

The complete pipeline is:

```text
Original + Anonymized
        ↓
Contextual term embeddings
        ↓
Cosine similarity matrix
        ↓
Maximum-weight matching
        ↓
Similarity threshold
        ↓
Affected original terms M
        ↓
Information content IC(t)
        ↓
TPI
```

The main output is a content-preservation score for an original/anonymized pair.

This notebook remains an experimental evaluation framework. It does not assume that
the semantic alignment or TPI formulation is optimal.

## Relation to the previous notebooks

### 07a / 07b — TPI

For an original transcript $W_o$:

$$
IC(t)=-\log P(t)
$$

and

$$
TPI
=
1-
\frac{
\sum_{t\in M} IC(t)
}{
\sum_{t\in W_o} IC(t)
}.
$$

The key difficulty is obtaining the affected set $M$.

### 07c — Surface alignment

07c used `SequenceMatcher` to obtain $M$, but lexical matching can incorrectly
flag paraphrases and changes in word order.

### 07d — Semantic alignment

07d replaced this surface alignment with contextual BERT embeddings and
maximum-weight one-to-one matching.

This notebook connects the two stages:

$$
\boxed{
\text{Semantic alignment}
\;\rightarrow\;
M
\;\rightarrow\;
\text{TPI}
}
$$

The threshold remains an explicit experimental parameter.

In [1]:
import math
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from scipy.optimize import linear_sum_assignment

print("PyTorch:", torch.__version__)

/opt/anaconda3/envs/interview-anon/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.11.0


In [2]:
MODEL_NAME = "bert-base-uncased"

# Current experimental value from the 07d controlled tests.
SIMILARITY_THRESHOLD = 0.80

TPI_STRATEGY = "N-TA"
N = 6

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# One MLM model provides both contextual hidden states and masked-token probabilities.
model = AutoModelForMaskedLM.from_pretrained(
    MODEL_NAME,
    output_hidden_states=True
).to(DEVICE)

model.eval()

print("Model:", MODEL_NAME)
print("Device:", DEVICE)
print("Semantic similarity threshold:", SIMILARITY_THRESHOLD)
print("TPI strategy:", TPI_STRATEGY)
print("N-TA N:", N)

Loading weights: 100%|█████████████████████| 202/202 [00:00<00:00, 4588.73it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: bert-base-uncased
Device: cpu
Semantic similarity threshold: 0.8
TPI strategy: N-TA
N-TA N: 6


## 1. Term extraction

We keep the same lexical term definition used in 07a/07c/07d.

Punctuation is ignored, while each term keeps its character span.

In [3]:
TERM_PATTERN = r"[A-Za-z0-9]+(?:['-][A-Za-z0-9]+)*"

def extract_terms(text):
    return [
        {"term": m.group(), "start": m.start(), "end": m.end()}
        for m in re.finditer(TERM_PATTERN, text)
    ]

def normalize_term(term):
    return term.lower()

def term_subtokens(term):
    return tokenizer.tokenize(term)

def show_terms(text):
    return pd.DataFrame(extract_terms(text))

show_terms("I moved to Geneva in 2022.")

,term,start,end
0,I,0,1
1,moved,2,7
2,to,8,10
3,Geneva,11,17
4,in,18,20
5,2022,21,25


## 2. Contextual embeddings for terms

As in 07d, BERT tokenizes the complete sentence. For each lexical term we
identify the WordPiece tokens overlapping its character span and mean-pool
their contextual hidden states:

$$
e_t=\frac{1}{K}\sum_{k=1}^{K}h_k.
$$

These embeddings are used for alignment, not directly as the information-loss
measure.

In [4]:
def term_embeddings(text):
    terms = extract_terms(text)

    if not terms:
        return terms, np.empty(
            (0, model.config.hidden_size),
            dtype=np.float32
        )

    encoded = tokenizer(
        text,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        max_length=tokenizer.model_max_length,
    )

    offsets = encoded.pop("offset_mapping")[0].tolist()
    encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)

    hidden = outputs.hidden_states[-1][0].detach().cpu()

    term_vectors = []

    for term in terms:
        overlapping_token_ids = []

        for token_id, (tok_start, tok_end) in enumerate(offsets):
            if tok_end <= tok_start:
                continue

            if tok_start < term["end"] and tok_end > term["start"]:
                overlapping_token_ids.append(token_id)

        if not overlapping_token_ids:
            raise ValueError(
                f"No tokenizer token found for term: {term['term']!r}"
            )

        vector = hidden[overlapping_token_ids].mean(dim=0)
        term_vectors.append(vector.numpy())

    return terms, np.vstack(term_vectors).astype(np.float32)

## 3. Semantic alignment

For every original/anonymized term pair:

$$
S_{ij}
=
\frac{e_i^{orig}\cdot e_j^{anon}}
{\|e_i^{orig}\|\|e_j^{anon}\|}.
$$

We solve a one-to-one maximum-weight matching problem:

$$
\max_A \sum_{(i,j)\in A}S_{ij}.
$$

Only matched pairs with $S_{ij}\geq	\tau$ are accepted.

Any original term that is not confidently matched is included in \(M\).

In [5]:
def cosine_similarity_matrix(A, B):
    A_norm = A / np.clip(
        np.linalg.norm(A, axis=1, keepdims=True),
        1e-12,
        None
    )
    B_norm = B / np.clip(
        np.linalg.norm(B, axis=1, keepdims=True),
        1e-12,
        None
    )
    return A_norm @ B_norm.T


def build_similarity_matrix(original_text, anonymized_text):
    original_terms, original_embeddings = term_embeddings(original_text)
    anonymized_terms, anonymized_embeddings = term_embeddings(anonymized_text)

    if len(original_terms) == 0 or len(anonymized_terms) == 0:
        similarity = np.empty(
            (len(original_terms), len(anonymized_terms))
        )
    else:
        similarity = cosine_similarity_matrix(
            original_embeddings,
            anonymized_embeddings
        )

    return original_terms, anonymized_terms, similarity


def semantic_align(
    original_text,
    anonymized_text,
    similarity_threshold=SIMILARITY_THRESHOLD
):
    original_terms, anonymized_terms, similarity = build_similarity_matrix(
        original_text, anonymized_text
    )

    n = len(original_terms)
    m = len(anonymized_terms)

    alignments = []
    matched_original = set()
    matched_anonymized = set()

    if n > 0 and m > 0:
        row_ind, col_ind = linear_sum_assignment(-similarity)

        for i, j in zip(row_ind, col_ind):
            score = float(similarity[i, j])

            if score >= similarity_threshold:
                matched_original.add(i)
                matched_anonymized.add(j)

                same_surface = (
                    normalize_term(original_terms[i]["term"])
                    == normalize_term(anonymized_terms[j]["term"])
                )

                status = (
                    "exact_match"
                    if same_surface
                    else "semantic_match"
                )

                alignments.append({
                    "original_idx": i,
                    "original_term": original_terms[i]["term"],
                    "anonymized_idx": j,
                    "anonymized_term": anonymized_terms[j]["term"],
                    "similarity": score,
                    "status": status,
                })

    affected_original_indices = [
        i for i in range(n)
        if i not in matched_original
    ]

    alignments = sorted(
        alignments,
        key=lambda x: x["original_idx"]
    )

    return {
        "original_terms": original_terms,
        "anonymized_terms": anonymized_terms,
        "similarity_matrix": similarity,
        "alignments": alignments,
        "affected_original_indices": affected_original_indices,
        "affected_original_terms": [
            original_terms[i]["term"]
            for i in affected_original_indices
        ],
        "unmatched_anonymized_indices": [
            j for j in range(m)
            if j not in matched_anonymized
        ],
    }


def alignment_dataframe(result):
    rows = []

    for item in result["alignments"]:
        rows.append({
            "original_idx": item["original_idx"],
            "original_term": item["original_term"],
            "anonymized_idx": item["anonymized_idx"],
            "anonymized_term": item["anonymized_term"],
            "similarity": round(item["similarity"], 4),
            "status": item["status"],
        })

    for i in result["affected_original_indices"]:
        rows.append({
            "original_idx": i,
            "original_term": result["original_terms"][i]["term"],
            "anonymized_idx": None,
            "anonymized_term": None,
            "similarity": None,
            "status": "affected",
        })

    return (
        pd.DataFrame(rows)
        .sort_values("original_idx")
        .reset_index(drop=True)
    )

## 4. Information content for the original transcript

TPI is computed only from the **original** transcript.

For a term $t$:

$$
IC(t)=-\log P(t\mid context).
$$

As in 07a, a multi-subtoken term uses the minimum probability across its
WordPiece subtokens.

The $N$-TA strategy masks terms in alternating groups.

In [6]:
def build_masked_text(text, target_indices):
    terms = extract_terms(text)
    target_indices = set(target_indices)

    pieces = []
    cursor = 0
    mask_lengths = {}

    for i, item in enumerate(terms):
        pieces.append(text[cursor:item["start"]])

        if i in target_indices:
            subtokens = term_subtokens(item["term"])
            pieces.append(
                " ".join([tokenizer.mask_token] * len(subtokens))
            )
            mask_lengths[i] = len(subtokens)
        else:
            pieces.append(text[item["start"]:item["end"]])

        cursor = item["end"]

    pieces.append(text[cursor:])

    return "".join(pieces), mask_lengths


def masked_term_probabilities(text, target_indices):
    terms = extract_terms(text)

    if not target_indices:
        return {}

    masked_text, _ = build_masked_text(text, target_indices)

    inputs = tokenizer(
        masked_text,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        logits = model(**inputs).logits[0]
        probabilities = torch.softmax(logits, dim=-1)

    input_ids = inputs["input_ids"][0]

    mask_positions = [
        i
        for i, token_id in enumerate(input_ids)
        if int(token_id) == tokenizer.mask_token_id
    ]

    results = {}
    cursor = 0

    for idx in sorted(target_indices):
        term = terms[idx]["term"]
        original_subtokens = term_subtokens(term)
        n_subtokens = len(original_subtokens)

        positions = mask_positions[cursor:cursor + n_subtokens]

        if len(positions) != n_subtokens:
            raise RuntimeError(
                f"Could not align masks for term '{term}'."
            )

        token_probs = []

        for position, subtoken in zip(
            positions, original_subtokens
        ):
            token_id = tokenizer.convert_tokens_to_ids(subtoken)
            token_probs.append(
                float(probabilities[position, token_id].item())
            )

        results[idx] = {
            "term": term,
            "subtokens": original_subtokens,
            "subtoken_probabilities": token_probs,
            "probability": min(token_probs),
        }

        cursor += n_subtokens

    return results


def information_content(probability):
    probability = max(float(probability), 1e-24)
    return -math.log(probability)

In [7]:
def estimate_all_term_ic(text, strategy=TPI_STRATEGY, N=N):
    terms = extract_terms(text)
    n_terms = len(terms)

    if n_terms == 0:
        return pd.DataFrame(
            columns=["index", "term", "probability", "IC"]
        )

    if strategy == "S-TA":
        groups = [[i] for i in range(n_terms)]
    elif strategy == "N-TA":
        groups = [
            list(range(offset, n_terms, N))
            for offset in range(N)
            if offset < n_terms
        ]
    else:
        raise ValueError(
            "strategy must be 'S-TA' or 'N-TA'."
        )

    records = []

    for group in groups:
        group_results = masked_term_probabilities(text, group)

        for idx in group:
            r = group_results[idx]
            records.append({
                "index": idx,
                "term": r["term"],
                "probability": r["probability"],
                "IC": information_content(r["probability"]),
            })

    return (
        pd.DataFrame(records)
        .sort_values("index")
        .reset_index(drop=True)
    )

## 5. TPI from the semantic-alignment output

Once semantic alignment identifies $M$, TPI is computed as:

$$
TPI
=
1-
\frac{
\sum_{t\in M}IC(t)
}{
\sum_{t\in W_o}IC(t)
}.
$$

We also report

$$
L_{TPI}=1-TPI
$$

for compatibility with a future loss formulation.

In [8]:
def tpi_from_alignment(
    original_text,
    alignment_result,
    strategy=TPI_STRATEGY,
    N=N
):
    ic_table = estimate_all_term_ic(
        original_text,
        strategy=strategy,
        N=N
    )

    affected_indices = alignment_result[
        "affected_original_indices"
    ]

    affected = ic_table[
        ic_table["index"].isin(affected_indices)
    ].copy()

    tic_original = float(ic_table["IC"].sum())
    information_lost = float(affected["IC"].sum())

    if tic_original <= 0:
        raise ValueError(
            "TIC(original) must be strictly positive."
        )

    tpi = 1.0 - information_lost / tic_original
    tpi = max(0.0, min(1.0, tpi))

    return {
        "TIC_original": tic_original,
        "information_lost": information_lost,
        "TPI": tpi,
        "L_TPI": 1.0 - tpi,
        "affected_terms": affected,
        "IC_table": ic_table,
    }

## 6. Complete semantic-alignment + TPI evaluation

This is the main interface of the notebook.

$$
\boxed{
\text{Original + Anonymized}
\;\rightarrow\;
\text{Semantic alignment}
\;\rightarrow\;
M
\;\rightarrow\;
TPI
}
$$

In [9]:
def evaluate_content_pair(
    original_text,
    anonymized_text,
    similarity_threshold=SIMILARITY_THRESHOLD,
    tpi_strategy=TPI_STRATEGY,
    N=N
):
    alignment_result = semantic_align(
        original_text,
        anonymized_text,
        similarity_threshold=similarity_threshold
    )

    tpi_result = tpi_from_alignment(
        original_text,
        alignment_result,
        strategy=tpi_strategy,
        N=N
    )

    return {
        "original_text": original_text,
        "anonymized_text": anonymized_text,
        "alignment": alignment_result,
        "tpi": tpi_result,
        "TPI": tpi_result["TPI"],
        "L_TPI": tpi_result["L_TPI"],
    }

## 7. Controlled end-to-end experiments

These cases are inherited from 07d. The difference is that we now propagate
the detected affected terms all the way through TPI.

The purpose is to check the complete implementation before applying it to
real data.

In [10]:
cases = {
    "unchanged": (
        "I live in Geneva.",
        "I live in Geneva."
    ),
    "synonym_paraphrase": (
        "I live in Geneva.",
        "I reside in Geneva."
    ),
    "replacement": (
        "I live in Lausanne.",
        "I live in Geneva."
    ),
    "deletion": (
        "I live in Geneva since 2022.",
        "I live in Geneva."
    ),
    "insertion": (
        "I live in Geneva.",
        "I currently live in Geneva."
    ),
    "reordering": (
        "I live in Geneva.",
        "In Geneva I live."
    ),
    "generalization": (
        "I have two brothers.",
        "I have siblings."
    ),
    "multiple_changes": (
        "I moved to Geneva in 2022.",
        "I moved to Switzerland."
    ),
}

controlled_rows = []

for name, (original, anonymized) in cases.items():
    result = evaluate_content_pair(
        original,
        anonymized,
        similarity_threshold=SIMILARITY_THRESHOLD
    )

    controlled_rows.append({
        "case": name,
        "affected_terms": ", ".join(
            result["alignment"]["affected_original_terms"]
        ),
        "information_lost": round(
            result["tpi"]["information_lost"], 4
        ),
        "TPI": round(result["TPI"], 4),
        "L_TPI": round(result["L_TPI"], 4),
    })

controlled_results = pd.DataFrame(controlled_rows)
display(controlled_results)

,case,affected_terms,information_lost,TPI,L_TPI
0,unchanged,,0.0000,1.0000,0.0000
1,synonym_paraphrase,,0.0000,1.0000,0.0000
2,replacement,Lausanne,8.9839,0.4075,0.5925
3,deletion,"since, 2022",16.3266,0.4059,0.5941
4,insertion,,0.0000,1.0000,0.0000
5,reordering,"I, live, in, Geneva",12.5777,0.0000,1.0000
6,generalization,"two, brothers",3.4443,0.5126,0.4874
7,multiple_changes,"Geneva, in, 2022",19.5526,0.3249,0.6751


## 8. Inspect one complete example

The output below exposes the complete chain:

$$
\text{alignment}
\;\rightarrow\;
M
\;\rightarrow\;
IC(M)
\;\rightarrow\;
TPI.
$$

This is useful for debugging surprising scores on real anonymized data.

In [11]:
example_original = "I live in Lausanne."
example_anonymized = "I live in Geneva."

example_result = evaluate_content_pair(
    example_original,
    example_anonymized,
    similarity_threshold=SIMILARITY_THRESHOLD
)

print("ORIGINAL:")
print(example_original)

print("\nANONYMIZED:")
print(example_anonymized)

print("\nALIGNMENT:")
display(alignment_dataframe(example_result["alignment"]))

print("\nAFFECTED TERMS:")
print(example_result["alignment"]["affected_original_terms"])

print("\nAFFECTED INFORMATION:")
display(example_result["tpi"]["affected_terms"])

print("\nTIC(original):", example_result["tpi"]["TIC_original"])
print("Information lost:", example_result["tpi"]["information_lost"])
print("TPI:", example_result["TPI"])
print("L_TPI:", example_result["L_TPI"])

ORIGINAL:
I live in Lausanne.

ANONYMIZED:
I live in Geneva.

ALIGNMENT:


,original_idx,original_term,anonymized_idx,anonymized_term,similarity,status
0,0,I,0.0,I,0.9799,exact_match
1,1,live,1.0,live,0.9767,exact_match
2,2,in,2.0,in,0.9589,exact_match
3,3,Lausanne,NaN,NaN,NaN,affected



AFFECTED TERMS:
['Lausanne']

AFFECTED INFORMATION:


,index,term,probability,IC
3,3,Lausanne,0.000125,8.983905



TIC(original): 15.16280385915622
Information lost: 8.983904966107774
TPI: 0.40750371438177324
L_TPI: 0.5924962856182268


## 9. Sanity checks

The implementation should preserve the properties already checked in 07b.

In particular:

$$
M=\varnothing \Rightarrow TPI=1
$$

and adding affected terms cannot increase TPI.

In [12]:
unchanged_result = evaluate_content_pair(
    "I live in Geneva.",
    "I live in Geneva.",
    similarity_threshold=SIMILARITY_THRESHOLD
)

assert unchanged_result["alignment"]["affected_original_indices"] == []
assert np.isclose(unchanged_result["TPI"], 1.0)

print("✓ Unchanged example gives TPI = 1.")

✓ Unchanged example gives TPI = 1.


In [13]:
test_text = "I moved to Geneva in 2022."

ic_test = estimate_all_term_ic(
    test_text,
    strategy=TPI_STRATEGY,
    N=N
)

def tpi_for_indices(ic_table, indices):
    tic = float(ic_table["IC"].sum())
    affected_ic = float(
        ic_table[
            ic_table["index"].isin(set(indices))
        ]["IC"].sum()
    )
    return 1.0 - affected_ic / tic

tpi_empty = tpi_for_indices(ic_test, [])
tpi_one = tpi_for_indices(ic_test, [3])
tpi_two = tpi_for_indices(ic_test, [3, 4])

assert tpi_empty >= tpi_one >= tpi_two

print("TPI(M = ∅):", round(tpi_empty, 4))
print("TPI(M = {3}):", round(tpi_one, 4))
print("TPI(M = {3,4}):", round(tpi_two, 4))
print("✓ Monotonicity check passed.")

TPI(M = ∅): 1.0
TPI(M = {3}): 0.7538
TPI(M = {3,4}): 0.7506
✓ Monotonicity check passed.


## 10. Applying the pipeline to real data

The next stage is to evaluate paired original/anonymized transcripts.

Expected input:

| original | anonymized |
|---|---|
| original transcript 1 | anonymized transcript 1 |
| original transcript 2 | anonymized transcript 2 |
| ... | ... |

The function below keeps the intermediate affected terms as well as the TPI
score so that surprising examples can be inspected manually.

In [14]:
def evaluate_dataset(
    df,
    original_column="original",
    anonymized_column="anonymized",
    similarity_threshold=SIMILARITY_THRESHOLD,
    tpi_strategy=TPI_STRATEGY,
    N=N
):
    rows = []

    for row_idx, row in df.iterrows():
        original = str(row[original_column])
        anonymized = str(row[anonymized_column])

        result = evaluate_content_pair(
            original,
            anonymized,
            similarity_threshold=similarity_threshold,
            tpi_strategy=tpi_strategy,
            N=N
        )

        rows.append({
            "row_idx": row_idx,
            "TPI": result["TPI"],
            "L_TPI": result["L_TPI"],
            "TIC_original": result["tpi"]["TIC_original"],
            "information_lost": result["tpi"]["information_lost"],
            "n_affected_terms": len(
                result["alignment"]["affected_original_indices"]
            ),
            "affected_terms": result["alignment"][
                "affected_original_terms"
            ],
        })

    return pd.DataFrame(rows)

## 11. Interpretation and next step

At this stage, TPI is treated as an **information-preservation component**,
not as a complete semantic content metric.

The content evaluation therefore remains decomposed into:

$$
\boxed{
\text{Semantic Similarity}
+
\text{NLI}
+
\text{TPI}
}
$$

We do **not** fix arbitrary weights for a final combined loss yet.

The immediate next step is empirical evaluation on real original/anonymized
data and analysis of where the three components agree or disagree.

In particular, we want to determine whether TPI adds information that is not
captured by semantic similarity and NLI.